# Pure SAM3 Multi-Organism 3D Part Segmentation (No SDF)
### Species-Agnostic Anatomical Fin & Appendage Extraction Across 6 Undecimated Marine Organisms

This notebook implements **Pure Multi-View SAM3 Segmentation without Shape Diameter Function (SDF)** on full-resolution, undecimated 3D models (`img_mesh_*.glb`) across 6 marine organisms (Tuna, Mackerel, Shark, Killer Whale, Goldfish, and Dolphin).

### Pipeline Architecture:
1. **Multi-View 2D Projection**: Renders 20 camera view projections ($512 \times 512$) of the undecimated 3D mesh.
2. **Zero-Shot SAM3 Instance Segmentation**: Queries SAM3 with text prompts (`['tail', 'top fin', 'side fins']`).
3. **3D Surface Backprojection**: Accumulates multi-view visibility and detection votes directly onto 3D mesh faces ($V_{vis}(f)$).
4. **Mesh Graph Candidate Proposal**: Connects faces with positive SAM votes into contiguous candidate patches.
5. **Species-Agnostic Anatomical Classification** (from Hybrid Architecture):
   - **Tail Fin (Caudal)**: Posterior extremity clusters ($cx \ge 0.45$ or top SAM prompt `tail`).
   - **Top Fin (Dorsal)**: Sagittal midline clusters along $+Y$ ($cy > 0.02$, $|cz| < 0.18$, top SAM prompt `top fin`).
   - **Pectoral Fins (Side Fins)**: Bilateral flank pairs in anterior/mid body ($cx < 0.45$, $|cz| > 0.015$, top SAM prompt `side fins`).
   - **Main Body**: Seamlessly retains head, torso, belly, and trunk geometry.
6. **Morphological Gap Filling & Boundary Closure**: Topological majority-voting closure on face adjacency graph.
7. **Planar Hole Capping via CDT**: Constrained Delaunay Triangulation caps open root loops on fin submeshes.
8. **Export**: Saves all watertight submeshes (`body.glb`, `tail.glb`, `top_fins.glb`, `left_pectoral_fin.glb`, `right_pectoral_fin.glb`) into `generated_data/final_test_segmentation/<organism>/`.

In [ ]:
import os
import sys
import gc
import json
import time
from pathlib import Path
from collections import defaultdict
import numpy as np
import cv2
import torch
import trimesh
import networkx as nx
import triangle
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import pandas as pd
from PIL import Image
from dotenv import load_dotenv

# Add repository root to path
REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv()

from animgen.core.models.model import BaseModelClass
from animgen.utils.mesh import triangle_areas
from animgen.rigging.SAM3 import SAM3Segmentation
from animgen.rigging.backproject import backproject_masks_to_faces

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Compute device: {DEVICE}")

## 1. Undecimated Organism Definitions & Output Paths

In [ ]:
OUTPUT_ROOT = REPO_ROOT / "generated_data" / "final_test_segmentation"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PROMPTS = [
    "tail",
    "top fin",
    "side fins",
]

# Undecimated models (img_mesh_*.glb)
ORGANISMS = {
    "tuna": {
        "name": "Tuna",
        "mesh_path": REPO_ROOT / "generated_data" / "models" / "img_mesh_Tuna.glb",
        "output_dir": OUTPUT_ROOT / "tuna",
    },
    "mackeral": {
        "name": "Mackerel",
        "mesh_path": REPO_ROOT / "generated_data" / "models" / "img_mesh_Mackeral.glb",
        "output_dir": OUTPUT_ROOT / "mackeral",
    },
    "shark": {
        "name": "Shark",
        "mesh_path": REPO_ROOT / "generated_data" / "models" / "img_mesh_Shark.glb",
        "output_dir": OUTPUT_ROOT / "shark",
    },
    "killer_whale": {
        "name": "Killer Whale",
        "mesh_path": REPO_ROOT / "generated_data" / "models" / "img_mesh_Killer_Whale.glb",
        "output_dir": OUTPUT_ROOT / "killer_whale",
    },
    "goldfish": {
        "name": "Goldfish",
        "mesh_path": REPO_ROOT / "generated_data" / "models" / "img_mesh_Goldfish.glb",
        "output_dir": OUTPUT_ROOT / "goldfish",
    },
    "dolphin": {
        "name": "Dolphin",
        "mesh_path": REPO_ROOT / "generated_data" / "models" / "img_mesh_Dolphin.glb",
        "output_dir": OUTPUT_ROOT / "dolphin",
    },
}

COLOR_MAP_NORM = {
    "Main Body": np.array([0.22, 0.25, 0.29]),
    "Top Fin": np.array([0.12, 0.63, 1.00]),         # Cyan
    "Tail Fin": np.array([0.96, 0.24, 0.24]),        # Red
    "Left Pectoral Fin": np.array([0.16, 0.86, 0.35]), # Green
    "Right Pectoral Fin": np.array([1.00, 0.70, 0.00]),# Amber/Orange
}

## 2. Planar Hole Capping via Constrained Delaunay Triangulation (CDT)

In [ ]:
def cap_root_hole_with_triangle(submesh: trimesh.Trimesh) -> trimesh.Trimesh:
    """Caps planar root boundary loops using Constrained Delaunay Triangulation (CDT)."""
    submesh = submesh.copy()
    edges = submesh.faces[:, [0, 1, 1, 2, 2, 0]].reshape(-1, 2)
    edges_sorted = np.sort(edges, axis=1)
    unique_edges, unique_inverse, counts = np.unique(
        edges_sorted, axis=0, return_inverse=True, return_counts=True
    )
    boundary_edge_mask = counts[unique_inverse] == 1
    boundary_directed = edges[boundary_edge_mask]

    if len(boundary_directed) == 0:
        return submesh

    G = nx.DiGraph()
    for u, v in boundary_directed:
        G.add_edge(u, v)

    loops = [c for c in nx.simple_cycles(G) if len(c) >= 3]
    if not loops:
        return submesh

    all_verts = list(submesh.vertices)
    all_faces = list(submesh.faces)

    for loop_vert_indices in loops:
        loop_vert_indices = np.array(loop_vert_indices, dtype=np.int64)
        loop_pts_3d = submesh.vertices[loop_vert_indices]
        K = len(loop_pts_3d)
        if K < 3:
            continue

        centroid = loop_pts_3d.mean(axis=0)
        centered = loop_pts_3d - centroid
        _, _, vh = np.linalg.svd(centered)
        u1, u2 = vh[0], vh[1]
        normal = np.cross(u1, u2)
        norm_len = np.linalg.norm(normal)
        if norm_len > 1e-12:
            normal = normal / norm_len

        pts_2d = np.column_stack([np.dot(centered, u1), np.dot(centered, u2)])
        segments = np.column_stack([np.arange(K), np.roll(np.arange(K), -1)])

        try:
            tri_out = triangle.triangulate({'vertices': pts_2d, 'segments': segments}, 'p')
        except Exception:
            try:
                tri_out = triangle.triangulate({'vertices': pts_2d, 'segments': segments}, 'c')
            except Exception:
                continue

        out_verts_2d = tri_out['vertices']
        out_triangles = tri_out['triangles']

        vert_map = {i: loop_vert_indices[i] for i in range(K)}
        for i in range(K, len(out_verts_2d)):
            p2 = out_verts_2d[i]
            v3d = centroid + p2[0] * u1 + p2[1] * u2
            vert_map[i] = len(all_verts)
            all_verts.append(v3d)

        for tri in out_triangles:
            mapped_tri = [vert_map[tri[0]], vert_map[tri[1]], vert_map[tri[2]]]
            p0 = all_verts[mapped_tri[0]]
            p1 = all_verts[mapped_tri[1]]
            p2 = all_verts[mapped_tri[2]]
            tri_norm = np.cross(p1 - p0, p2 - p0)
            if np.dot(tri_norm, normal) < 0:
                mapped_tri = [mapped_tri[0], mapped_tri[2], mapped_tri[1]]
            all_faces.append(mapped_tri)

    return trimesh.Trimesh(vertices=np.array(all_verts), faces=np.array(all_faces), process=True)

## 3. Morphological Gap Filling & Boundary Closure

In [ ]:
def fill_face_gaps(mesh: trimesh.Trimesh, face_labels: np.ndarray, adj_dict: dict, max_iters: int = 2) -> np.ndarray:
    """Fills isolated face gaps using majority-voting across neighbor faces on the adjacency graph."""
    labels = face_labels.copy()
    for _ in range(max_iters):
        changed = 0
        for f in range(len(mesh.faces)):
            curr_lbl = labels[f]
            neighbors = adj_dict[f]
            if not neighbors:
                continue
            nb_labels = [labels[nb] for nb in neighbors]
            majority_lbl = max(set(nb_labels), key=nb_labels.count)
            if nb_labels.count(majority_lbl) >= len(neighbors) * 0.7 and majority_lbl != curr_lbl:
                labels[f] = majority_lbl
                changed += 1
        if changed == 0:
            break
    return labels

## 4. Multi-View SAM3 Inference (with Automatic VRAM Release)

In [ ]:
def run_sam3_multi_view(mesh_model: BaseModelClass, prompts: list[str] = PROMPTS):
    """Runs multi-view SAM3 segmentation and backprojects votes to 3D mesh faces."""
    with SAM3Segmentation(prompts=prompts) as sam3:
        masks_dict = sam3(mesh_model, threshold=0.5, mask_threshold=0.5)
        face_prompt_detected = backproject_masks_to_faces(
            masks_dict,
            mesh_model.views_output["faces"],
            len(mesh_model.mesh.faces),
        )
    return face_prompt_detected, masks_dict

## 5. Pure SAM Anatomical Classification (No SDF)

In [ ]:
def classify_fish_appendages_pure_sam(
    mesh: trimesh.Trimesh,
    raw_clusters: list[np.ndarray],
    total_mesh_area: float,
    face_prompt_detected: dict[str, np.ndarray],
) -> dict[str, list[np.ndarray]]:
    """
    Universal 4-part anatomical fin classification pipeline using purely SAM votes and spatial centroids (NO SDF).
    """
    face_areas = triangle_areas(mesh.vertices, mesh.faces)
    MAJOR_AREA_THRESHOLD = 0.0010  # 0.10% minimum mesh area
    x_min, x_max = float(mesh.bounds[0, 0]), float(mesh.bounds[1, 0])
    x_mid = 0.5 * (x_min + x_max)

    cluster_meta = []
    for comp in raw_clusters:
        c_area = float(np.sum(face_areas[comp]))
        pct = (c_area / total_mesh_area) * 100.0
        if pct < MAJOR_AREA_THRESHOLD:
            continue
        c_verts = np.unique(mesh.faces[comp])
        cent = mesh.vertices[c_verts].mean(axis=0)
        max_abs_z = float(np.max(np.abs(mesh.vertices[c_verts, 2])))
        span_y = float(mesh.vertices[c_verts, 1].max() - mesh.vertices[c_verts, 1].min())
        span_z = float(mesh.vertices[c_verts, 2].max() - mesh.vertices[c_verts, 2].min())

        prompt_votes = {p: int(np.sum(face_prompt_detected[p][comp])) for p in PROMPTS}
        top_prompt = max(prompt_votes, key=prompt_votes.get) if max(prompt_votes.values()) > 0 else "None"

        cluster_meta.append({
            "faces": comp,
            "area_pct": pct,
            "centroid": cent,
            "max_abs_z": max_abs_z,
            "span_y": span_y,
            "span_z": span_z,
            "top_prompt": top_prompt,
            "prompt_votes": prompt_votes,
            "label": None,
        })

    # Pass 1: Caudal Tail Fin (All posterior extremity clusters at x >= 0.45 or posterior with tail votes)
    for c in cluster_meta:
        cx, cy, cz = c["centroid"]
        if cx >= 0.45 or (cx > x_mid and (c["top_prompt"] == "tail" or c["prompt_votes"].get("tail", 0) > 0)):
            c["label"] = "Tail Fin"

    # Pass 2: Top / Dorsal Fin (Elevated clusters along +Y, sagittal center, NOT at tail terminus)
    for c in cluster_meta:
        if c["label"] is not None:
            continue
        cx, cy, cz = c["centroid"]
        if cy > 0.02 and abs(cz) < 0.18 and (c["top_prompt"] == "top fin" or cy > 0.08):
            c["label"] = "Top Fin"

    # Pass 3: Bilateral Pectoral Fins (Side Fins in anterior/mid body: cx < 0.45)
    unlabeled = [c for c in cluster_meta if c["label"] is None and c["centroid"][0] < 0.45]

    left_candidates = [c for c in unlabeled if c["centroid"][2] > 0.015]
    right_candidates = [c for c in unlabeled if c["centroid"][2] < -0.015]

    left_candidates.sort(key=lambda c: (c["prompt_votes"].get("side fins", 0), c["max_abs_z"]), reverse=True)
    right_candidates.sort(key=lambda c: (c["prompt_votes"].get("side fins", 0), c["max_abs_z"]), reverse=True)

    for i, c in enumerate(left_candidates):
        if i == 0 and (c["max_abs_z"] > 0.06 or c["prompt_votes"].get("side fins", 0) > 0):
            c["label"] = "Left Pectoral Fin"
        else:
            c["label"] = "Main Body"

    for i, c in enumerate(right_candidates):
        if i == 0 and (c["max_abs_z"] > 0.06 or c["prompt_votes"].get("side fins", 0) > 0):
            c["label"] = "Right Pectoral Fin"
        else:
            c["label"] = "Main Body"

    classified = defaultdict(list)
    for c in cluster_meta:
        if c["label"] and c["label"] != "Main Body":
            print(f"   [Classified] {c['label']:20s} ({c['area_pct']:5.2f}% area) | cent=({c['centroid'][0]:+.2f}, {c['centroid'][1]:+.2f}, {c['centroid'][2]:+.2f}) | Top SAM Prompt: '{c['top_prompt']}'")
            classified[c["label"]].append(c["faces"])

    return classified

def render_mesh_4views_shaded(mesh: trimesh.Trimesh, face_colors_rgb: np.ndarray, title: str, save_path: Path):
    """Renders 4-view shaded figure of segmented 3D mesh."""
    verts = mesh.vertices
    faces = mesh.faces

    face_normals = mesh.face_normals
    light_dir = np.array([0.4, 0.6, 0.9])
    light_dir /= np.linalg.norm(light_dir)
    diffuse = np.clip(np.dot(face_normals, light_dir), 0.25, 1.0)
    shaded_colors = face_colors_rgb * diffuse[:, None]

    triangles = verts[faces]

    fig = plt.figure(figsize=(18, 5))
    fig.patch.set_facecolor("#0F1318")
    fig.suptitle(title, fontsize=15, color="#E8EDF5", fontweight="bold", y=0.98)

    views = [
        ("Side View (Lateral)", 0, 0),
        ("Perspective Angled", 25, -45),
        ("Top View (Dorsal)", 90, -90),
        ("Quarter View", -20, 135),
    ]

    for idx, (view_name, elev, azim) in enumerate(views, 1):
        ax = fig.add_subplot(1, 4, idx, projection="3d")
        ax.set_facecolor("#151A21")
        mesh_collection = Poly3DCollection(
            triangles, facecolors=shaded_colors, edgecolors="none", alpha=1.0
        )
        ax.add_collection3d(mesh_collection)

        max_range = (verts.max(axis=0) - verts.min(axis=0)).max() / 2.0
        mid = (verts.max(axis=0) + verts.min(axis=0)) * 0.5
        ax.set_xlim(mid[0] - max_range, mid[0] + max_range)
        ax.set_ylim(mid[1] - max_range, mid[1] + max_range)
        ax.set_zlim(mid[2] - max_range, mid[2] + max_range)

        ax.view_init(elev=elev, azim=azim)
        ax.axis("off")
        ax.set_title(view_name, color="#8AB4F8", fontsize=11, pad=-5)

    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

## 6. Single Organism Segmentation Pipeline Function

In [ ]:
def segment_organism_pure_sam(org_key: str, org_info: dict):
    print("\n" + "=" * 70)
    print(f"PROCESSING UNDECIMATED ORGANISM (PURE SAM - NO SDF): {org_info['name']}")
    print(f"Mesh path: {org_info['mesh_path']}")
    print("=" * 70)

    t0 = time.time()
    mesh_model = BaseModelClass(org_info["mesh_path"], renderer_size=(512, 512))
    trimesh_obj = mesh_model.mesh
    num_faces = len(trimesh_obj.faces)
    num_verts = len(trimesh_obj.vertices)
    face_areas = triangle_areas(trimesh_obj.vertices, trimesh_obj.faces)
    total_mesh_area = float(np.sum(face_areas))

    # 1. SAM3 Multi-View Inference
    print(f"1. Running SAM3 Multi-View Inference on Undecimated {num_faces:,} faces (20 Views)...")
    face_prompt_detected, _ = run_sam3_multi_view(mesh_model)
    total_sam_votes = np.sum(list(face_prompt_detected.values()), axis=0)

    # 2. Adjacency Graph Setup
    adj = trimesh_obj.face_adjacency
    adj_dict = defaultdict(list)
    for f1, f2 in adj:
        adj_dict[f1].append(f2)
        adj_dict[f2].append(f1)

    # 3. Candidate Proposal based directly on SAM detection
    is_sam_candidate = total_sam_votes >= 1
    face_centroids = trimesh_obj.triangles.mean(axis=1)
    x_min, x_max = float(trimesh_obj.bounds[0, 0]), float(trimesh_obj.bounds[1, 0])
    snout_thresh = x_min + 0.12 * (x_max - x_min)
    is_snout = (face_centroids[:, 0] < snout_thresh) & (np.abs(face_centroids[:, 2]) < 0.12) & (np.abs(face_centroids[:, 1]) < 0.12)
    candidate_mask = is_sam_candidate & (~is_snout)

    visited = np.zeros(num_faces, dtype=bool)
    raw_clusters = []
    for f in np.where(candidate_mask)[0]:
        if visited[f]:
            continue
        comp = []
        queue = [f]
        visited[f] = True
        while queue:
            curr = queue.pop()
            comp.append(curr)
            for nb in adj_dict[curr]:
                if candidate_mask[nb] and not visited[nb]:
                    visited[nb] = True
                    queue.append(nb)
        raw_clusters.append(np.array(comp, dtype=np.int32))

    raw_clusters.sort(key=lambda c: float(np.sum(face_areas[c])), reverse=True)
    print(f"2. Found {len(raw_clusters)} SAM-derived candidate clusters.")

    # 4. Anatomical Classification
    print("3. Applying Anatomical Appendage Classification...")
    classified_appendages = classify_fish_appendages_pure_sam(
        trimesh_obj, raw_clusters, total_mesh_area, face_prompt_detected
    )

    # 5. Part Assembly & Morphological Gap Filling
    face_label_array = np.zeros(num_faces, dtype=np.int32)
    label_to_id = {"Main Body": 0}
    id_to_label = {0: "Main Body"}

    part_id = 1
    for label, comp_list in classified_appendages.items():
        label_to_id[label] = part_id
        id_to_label[part_id] = label
        for comp in comp_list:
            face_label_array[comp] = part_id
        part_id += 1

    print("4. Applying Morphological Gap Filling across Face Boundaries...")
    refined_face_labels = fill_face_gaps(trimesh_obj, face_label_array, adj_dict, max_iters=2)

    final_appendages = {}
    for pid, label in id_to_label.items():
        if pid == 0:
            continue
        p_faces = np.where(refined_face_labels == pid)[0]
        if len(p_faces) == 0:
            continue
        final_appendages[label] = {
            "faces": p_faces,
            "area": float(np.sum(face_areas[p_faces])),
            "centroid": trimesh_obj.vertices[np.unique(trimesh_obj.faces[p_faces])].mean(axis=0),
            "verts": np.unique(trimesh_obj.faces[p_faces]),
        }

    # Main Body
    body_faces = np.where(refined_face_labels == 0)[0]
    final_appendages["Main Body"] = {
        "faces": body_faces,
        "area": float(np.sum(face_areas[body_faces])),
        "centroid": trimesh_obj.vertices[np.unique(trimesh_obj.faces[body_faces])].mean(axis=0),
        "verts": np.unique(trimesh_obj.faces[body_faces]),
    }

    # 6. Submesh Export with Planar Hole Capping
    out_dir = org_info["output_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)

    def get_filename(lbl):
        l_lower = lbl.lower()
        if "body" in l_lower:
            return "body.glb"
        elif "tail" in l_lower or "caudal" in l_lower:
            return "tail.glb"
        elif "top" in l_lower or "dorsal" in l_lower:
            return "top_fins.glb"
        elif "left" in l_lower and "pectoral" in l_lower:
            return "left_pectoral_fin.glb"
        elif "right" in l_lower and "pectoral" in l_lower:
            return "right_pectoral_fin.glb"
        else:
            return f"{l_lower.replace(' ', '_')}.glb"

    print(f"\n5. Exporting Watertight Submeshes to {out_dir}:")
    exported_summary = {}
    for label, data in final_appendages.items():
        blob_faces = data["faces"]
        subm = trimesh_obj.submesh([blob_faces], append=True)
        if "Main Body" not in label:
            capped_subm = cap_root_hole_with_triangle(subm)
        else:
            capped_subm = subm

        fname = get_filename(label)
        out_path = out_dir / fname
        capped_subm.export(out_path)

        area_pct = (data["area"] / total_mesh_area) * 100.0
        print(f"   {label:25s} -> {fname:22s} | {len(capped_subm.vertices):5d} verts, {len(capped_subm.faces):5d} faces | {area_pct:5.2f}% area")
        exported_summary[label] = {
            "file": fname,
            "verts": len(capped_subm.vertices),
            "faces": len(capped_subm.faces),
            "area_pct": round(area_pct, 2),
        }

    # 7. Render 4-View Colored 3D Shaded Mesh
    face_colors = np.zeros((num_faces, 3), dtype=np.float32)
    face_colors[:] = COLOR_MAP_NORM["Main Body"]
    for label, data in final_appendages.items():
        if label in COLOR_MAP_NORM:
            face_colors[data["faces"]] = COLOR_MAP_NORM[label]

    render_path = out_dir / f"pure_sam_segmented_4view_{org_key}.png"
    render_mesh_4views_shaded(
        mesh=trimesh_obj,
        face_colors_rgb=face_colors,
        title=f"Pure SAM Segmentation (No SDF): {org_info['name']} ({num_faces:,} Faces, {len(final_appendages)} Parts)",
        save_path=render_path,
    )

    elapsed = time.time() - t0
    print(f"Completed {org_info['name']} in {elapsed:.2f}s. Render saved to {render_path.name}")

    return exported_summary, final_appendages, trimesh_obj

## 7. Execution Across All 6 Species

In [ ]:
all_summaries = {}
all_mesh_data = {}

for org_key, org_info in ORGANISMS.items():
    if not org_info["mesh_path"].exists():
        print(f"Skipping {org_info['name']}, mesh not found at {org_info['mesh_path']}")
        continue
    summary, appendages, mesh_obj = segment_organism_pure_sam(org_key, org_info)
    all_summaries[org_key] = summary
    all_mesh_data[org_key] = {
        "mesh": mesh_obj,
        "appendages": appendages
    }

# Save Master Summary JSON
summary_json_path = OUTPUT_ROOT / "pure_sam_all_species_summary.json"
with open(summary_json_path, "w") as f:
    json.dump(all_summaries, f, indent=2)

print("\n" + "=" * 70)
print("ALL 6 UNDECIMATED SPECIES SEGMENTED SUCCESSFULLY WITH PURE SAM (NO SDF)!")
print(f"All submeshes and renders saved in: {OUTPUT_ROOT}")
print("=" * 70)

## 8. Comparative Visualizations across All 6 Species

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 15))
fig.patch.set_facecolor("#0F1318")
fig.suptitle("Pure SAM 3D Part Segmentation across 6 Marine Species (No SDF)", fontsize=18, color="#E8EDF5", fontweight="bold", y=0.98)

for idx, (org_key, org_info) in enumerate(ORGANISMS.items()):
    ax = axes[idx // 2, idx % 2]
    render_img_path = org_info["output_dir"] / f"pure_sam_segmented_4view_{org_key}.png"
    if render_img_path.exists():
        img = Image.open(render_img_path)
        ax.imshow(img)
    ax.axis("off")
    ax.set_title(org_info["name"], color="#8AB4F8", fontsize=13, pad=4)

plt.tight_layout()
plt.show()

## 9. Comprehensive Submesh Export Summary Table

In [ ]:
rows = []
for org_key, parts in all_summaries.items():
    org_name = ORGANISMS[org_key]["name"]
    for part_name, info in parts.items():
        rows.append({
            "Organism": org_name,
            "Anatomical Part": part_name,
            "Filename": info["file"],
            "Vertices": f"{info['verts']:,}",
            "Faces": f"{info['faces']:,}",
            "Area %": f"{info['area_pct']:.2f}%",
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))